In [ ]:
# ===== Settings: everything this notebook reads is set in this cell =====
DATASET = "eds"
SPLIT = "test"

# run_v2_fast folders to compare, one per sigma (relative to the repository root, or absolute).
# They must store the three sums, i.e. be made with the current run_v2_fast.
RUN_FOLDERS = [
    "MMD/runs/eds-s1-t1",
    "MMD/runs/eds-s2-t1",
    "MMD/runs/eds-s5-t1",
]

# Plot 3: one (sequence, generator) of one run, in a short time window
CLOUD_RUN_FOLDER = "MMD/runs/eds-s5-t1"
CLOUD_SEQUENCE = "02_rocket_earth_light"
CLOUD_GENERATOR = "evtween_v7_teacher"
CLOUD_START_S = None          # window start in seconds; None = middle of the sequence
CLOUD_DURATION_MS = 5.0       # keep it short: every pair of nearby events is summed
CLOUD_REGION = None           # (x, y, width, height) patch of the frame, or None for all of it
CLOUD_SIGMAS_PX = [1, 2, 5]   # spatial sigmas offered in the plot's dropdown
CLOUD_SIGMA_T_MS = 1.0        # temporal sigma
CLOUD_MAX_POINTS = 15000      # events drawn per stream (the sums always use all of them)

# Where Plot 3 reads events. None = the GT folder and prediction file the run recorded for
# that pair (what was scored). Otherwise roots with the --data-root / --pred-root layout.
GT_ROOT = None
PRED_ROOT = None

# MMD explorer (run_v2_fast results)

Per sequence, with $N$ GT and $M$ generated events and the Gaussian kernel $k$ (zero between ON and OFF):

$$\text{prob\_distance}^2 = \underbrace{\tfrac{1}{N^2}\textstyle\sum k(g,g')}_{a\ \text{GT–GT}} + \underbrace{\tfrac{1}{M^2}\sum k(p,p')}_{b\ \text{pred–pred}} - 2\,\underbrace{\tfrac{1}{NM}\sum k(g,p)}_{c\ \text{GT–pred}}, \qquad \text{similarity} = \frac{c}{\sqrt{a\,b}}$$

* **a** only depends on GT: how clustered GT events are at the kernel's scale.
* **b**: the same for the prediction. $b/a > 1$ means the prediction is more clustered than GT.
* **c**: how much the two overlap. Similarity rescales it to [0, 1].

Contents: **Table 1 / Plot 1** the metrics across sigma · **Table 2 / Plot 2** the three terms · **Plot 3** the terms event by event in a short window. All plots are interactive (dropdown at the top left, hover, zoom).

In [ ]:
import sys
import time
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "MMD" / "explore_v2_fast.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from MMD import explore_v2_fast as ex

runs = ex.runs_from_folders(RUN_FOLDERS, DATASET, SPLIT)
display(runs[["run", "sigma_spatial_px", "sigma_temporal_ms", "cutoff_sigma", "rows"]])
scores = ex.load_scores(runs)
print(f"{len(scores)} scored (sequence, generator) pairs; "
      f"a + b - 2c reproduces prob_distance^2 to {ex.check_terms(scores):.0e}")

## 1 · The metrics across sigma

**Table 1**: mean over sequences, per generator (rows) and sigma (columns). Similarity: higher is better. Rank (1 = best within a sequence), prob_distance and count_relative_to_empty: lower is better.

**Plot 1**: the same against spatial sigma; pick the metric in the dropdown. Error bars = spread (std) over sequences. A ranking that holds at every sigma is robust; lines that cross mean the order depends on the scale.

In [ ]:
display(ex.metric_table(scores).style.format(precision=4)
        .set_caption("Table 1 · mean over sequences"))

In [ ]:
ex.metric_vs_sigma(scores).show()

## 2 · The three terms

**Table 2**: mean over sequences of a, b, c, their combination a + b − 2c (= prob_distance²) and the ratios b/a, c/a.

**Plot 2**: a, b, c against spatial sigma, one panel each. *Absolute* (log scale) shows how all terms grow with sigma; *Relative to GT–GT* divides by a, so sequences with different event densities weigh the same: b/a is the prediction's clustering compared with GT (1 = same), c/a its overlap with GT.

In [ ]:
display(ex.terms_table(scores).style.format("{:.3g}")
        .set_caption("Table 2 · mean over sequences of the three terms"))

In [ ]:
ex.terms_vs_sigma(scores).show()

## 3 · Inside one short window

The three sums event by event, for the window set in the first cell (GT ○, prediction ◇; x, time, y):

* **GT – GT**: for each GT event, $\sum_j k(g_i, g_j)$, how many GT events lie within about one sigma (itself included).
* **prediction – prediction**: the same within the prediction.
* **GT – prediction**: around each GT event, the prediction's events (and the reverse for prediction events).
* **difference**: GT density minus prediction density at each event, both normalized. Red: GT has events the prediction lacks. Blue: the prediction puts events where GT has none. The window's distance² is exactly the red minus blue total.

The dropdown switches term and sigma; the title gives the window's a, b, c and similarity (for this window only, not the whole sequence). If it takes long, shorten `CLOUD_DURATION_MS` or set `CLOUD_REGION`.

In [ ]:
started = time.perf_counter()
window = ex.load_window(CLOUD_RUN_FOLDER, CLOUD_SEQUENCE, CLOUD_GENERATOR, CLOUD_START_S,
                        CLOUD_DURATION_MS, CLOUD_REGION, GT_ROOT, PRED_ROOT)
print(f"{CLOUD_SEQUENCE} / {CLOUD_GENERATOR}: {len(window['gt'].t):,} GT and "
      f"{len(window['generated'].t):,} generated events in "
      f"{window['start_s']:.4f}-{window['stop_s']:.4f} s")
cloud, cloud_terms = ex.point_cloud(window, CLOUD_SIGMAS_PX, CLOUD_SIGMA_T_MS, CLOUD_MAX_POINTS)
print(f"computed in {time.perf_counter() - started:.1f} s")
cloud.show()